# WOCU erosion prediction — master pipeline

This notebook runs the full pipeline end-to-end by calling functions from `src/pipeline/`.
Each step saves its output to the experiment's data folder so you can re-run individual
steps independently.

```
01_raw/erosion/*.gpkg
        │
        ▼
  [Step 1] bank_distances.compute_bank_distances()
        │  → 03_features/{EXPERIMENT}/dist_per_year.parquet
        ▼
  [Step 2] region_split.build_region_split()
        │  → 03_features/{EXPERIMENT}/region_split.parquet
        │  → 03_features/{EXPERIMENT}/region_inference_only.parquet
        ▼
  [Step 3] feature_engineering.build_features()
        │  → 03_features/{EXPERIMENT}/region_features.parquet
        │  → 03_features/{EXPERIMENT}/region_inference_features.parquet
        ▼
  [Step 4] train.train_and_save_models()
        │  → 04_model_outputs/{EXPERIMENT}/bundle.joblib
        │  → 04_model_outputs/{EXPERIMENT}/model_*.joblib
        ▼
  [Step 5] predictor.predict_iterative_ml()  +  export.export_predictions()
           → 04_model_outputs/{EXPERIMENT}/wocu_lgb_predictions_{EXPERIMENT}.gpkg
```

To run a **new experiment**, duplicate this notebook folder, change `EXPERIMENT` below,
and re-run all cells. Everything else is driven by that single string.

## 0. Setup

In [1]:
import os, sys
from pathlib import Path

_cwd = Path.cwd()
for _candidate in [_cwd, *_cwd.parents]:
    if (_candidate / 'src').exists():
        _backend = _candidate
        break
else:
    raise RuntimeError('Could not find backend root (directory containing src/)')

os.chdir(_backend)
if str(_backend) not in sys.path:
    sys.path.insert(0, str(_backend))

print('backend root:', _backend)

backend root: /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend


In [2]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import geopandas as gpd
from pyogrio import list_layers

import src.paths as PATHS
from src.pipeline.bank_distances    import compute_bank_distances
from src.pipeline.region_split      import build_region_split
from src.pipeline.feature_engineering import build_features
from src.pipeline.train             import train_and_save_models
from src.model.export_utils         import load_model_bundle
from src.model.predictor            import predict_iterative_ml
from src.erosion.centerline_utils   import (
    build_ref_geom_lookup, compute_vvr_crossing_year,
    ensure_location_id_column, get_nvo_location_ids, point_from_offset,
)
from src.erosion.export             import export_predictions

print('Imports OK')

Imports OK


In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
#  EXPERIMENT CONFIGURATION — change these for a new run
# ═══════════════════════════════════════════════════════════════════════════════

EXPERIMENT  = '20260617a'

# Input data (same as 20260314)
RAW_GPKG         = PATHS.DATA_DIR / '01_raw/erosion/wocu_output_fase2_20260210.gpkg'
PROC_GPKG        = PATHS.DATA_DIR / '02_processed/erosion/wocu_post_processed_fase2_20260310.gpkg'
SCOPE_GPKG       = PATHS.DATA_DIR / '01_raw/scope/scope_fase2.gpkg'
VEG_GPKG         = PATHS.DATA_DIR / '02_processed/wfs_context/vegetatielegger.gpkg'
LU_GPKG          = PATHS.DATA_DIR / '02_processed/wfs_context/land_use.gpkg'
SOIL_GPKG        = PATHS.DATA_DIR / '01_raw/soil/BRO_DownloadBodemkaart.gpkg'
STATIONS_GPKG    = PATHS.DATA_DIR / '02_processed/water_stations/water_stations_for_modeling.gpkg'
DISCHARGE_DIR    = PATHS.DATA_DIR / 'water_stations_timeseries/cleaned/discharge'
REF_FEATURES_V2  = PATHS.DATA_DIR / '02_processed/erosion/region_features_v2.parquet'
SIGNALERING_GPKG = PATHS.DATA_DIR / '01_raw/scope/20260205_signaleringslijn.gpkg'
SIGNALERING_LAYER = 'Vlak_vrije_ruimte_natuurvriendelijke_oever_ln'

# Pipeline parameters
N_POINTS   = 3      # top-N furthest OK points per region × year
TEST_SIZE  = 0.20
SEED       = 42
START_YEAR = 2026
END_YEAR   = 2050

# Output directories
FEATURES_DIR  = PATHS.DATA_DIR / f'03_features/{EXPERIMENT}'
MODEL_OUT_DIR = PATHS.DATA_DIR / f'04_model_outputs/{EXPERIMENT}'
FEATURES_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)

PREDICTION_YEARS = list(range(START_YEAR, END_YEAR + 1))

print(f'Experiment      : {EXPERIMENT}')
print(f'Features dir    : {FEATURES_DIR}')
print(f'Model output    : {MODEL_OUT_DIR}')
print(f'Prediction years: {START_YEAR}–{END_YEAR}')

Experiment      : 20260617a
Features dir    : /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/data/03_features/20260617a
Model output    : /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/data/04_model_outputs/20260617a
Prediction years: 2026–2050


## Step 1 — Bank distances

Reduces ~5M raw bank points to one representative `dist_m` per `(location_id, year)`.
Takes the mean of the N_POINTS furthest OK-status points per group.

In [4]:
print('Running Step 1: bank distances ...')
dist_per_year = compute_bank_distances(RAW_GPKG, n_points=N_POINTS)

dist_per_year.to_parquet(FEATURES_DIR / 'dist_per_year.parquet', index=False)

print(f'Shape  : {dist_per_year.shape}')
print(f'Locations: {dist_per_year["location_id"].nunique():,}')
print(f'Years    : {sorted(dist_per_year["year"].unique())}')
print(f'Saved  → {FEATURES_DIR / "dist_per_year.parquet"}')
dist_per_year.head()

Running Step 1: bank distances ...
Shape  : (30228, 5)
Locations: 10,484
Years    : [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2024), np.int64(2025)]
Saved  → /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/data/03_features/20260617a/dist_per_year.parquet


,location_id,year,dist_m,n_ok_pts,n_selected
0,ijssel1_l_0000_0010,2017,162.583333,97.0,3.0
1,ijssel1_l_0000_0010,2025,162.083333,110.0,3.0
2,ijssel1_l_0010_0020,2017,155.478645,72.0,3.0
3,ijssel1_l_0010_0020,2022,156.229750,106.0,3.0
4,ijssel1_l_0010_0020,2025,156.730486,87.0,3.0


## Step 2 — Region split

Pivots the distance table to per-region records with t1/t2/t3,
applies quality filter, joins is_nvo, computes erosion volume rates,
and produces a stratified 80/20 train/test split.

In [5]:
print('Running Step 2: region split ...')
region_split, region_inference = build_region_split(
    dist_per_year,
    proc_gpkg=PROC_GPKG,
    test_size=TEST_SIZE,
    random_seed=SEED,
)

region_split.to_parquet(FEATURES_DIR / 'region_split.parquet')
region_inference.to_parquet(FEATURES_DIR / 'region_inference_only.parquet')

print(f'region_split         : {region_split.shape}')
print(f'region_inference_only: {region_inference.shape}')
print(f'NVO regions (split)  : {region_split["is_nvo"].sum():,}')
print('Train/test split per cluster:')
print(region_split.groupby(['cluster', 'split']).size().unstack(fill_value=0).to_string())
print(f'Saved → {FEATURES_DIR}') 

Running Step 2: region split ...
region_split         : (7444, 18)
region_inference_only: (650, 10)
NVO regions (split)  : 1,985
Train/test split per cluster:
split      test  train
cluster               
ijssel1     362   1448
ijssel2      57    230
maas1        25    102
maas2        36    142
maas3       536   2145
nederrijn   240    958
rijn        233    930
Saved → /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/data/03_features/20260617a


## Step 3 — Feature engineering

Enriches the split tables with vegetation, land use, soil, hydrology,
bend exposure, erosion volume rate, and ordinal encodings.

Expected runtime: ~2–3 min (soil overlay is the slow step).

In [6]:
print('Running Step 3: feature engineering ...')
region_features, region_inference_features = build_features(
    region_split,
    region_inference,
    scope_gpkg=SCOPE_GPKG,
    veg_gpkg=VEG_GPKG,
    lu_gpkg=LU_GPKG,
    soil_gpkg=SOIL_GPKG,
    stations_gpkg=STATIONS_GPKG,
    discharge_dir=DISCHARGE_DIR,
    reference_features_v2=REF_FEATURES_V2,
)

region_features.to_parquet(FEATURES_DIR / 'region_features.parquet')
region_inference_features.to_parquet(FEATURES_DIR / 'region_inference_features.parquet')

print(f'region_features           : {region_features.shape}')
print(f'region_inference_features : {region_inference_features.shape}')
print(f'Columns: {list(region_features.columns)}')
print('\nNull counts (region_features):')
nulls = region_features.isnull().sum()
print(nulls[nulls > 0].to_string() or '  (none)')
print(f'Saved → {FEATURES_DIR}')

Running Step 3: feature engineering ...
Loading scope geometries ...
Assigning vegetation class ...
Assigning land use ...
Assigning soil group (spatial overlay, ~30s) ...
Assigning nearest discharge station ...
Computing high-water window metrics ...
Loading bend exposure from reference ...
region_features           : (7444, 29)
region_inference_features : (650, 25)
Columns: ['v_train', 'v_test', 'dist_t1', 'dist_t2', 'dist_t3', 'train_span_yr', 'test_span_yr', 'is_nvo', 'river', 'river_enc', 'vegetation_class', 'vegetation_class_enc', 'land_use', 'land_use_enc', 'erosion_vol_rate_t1', 'soil_group', 'soil_group_enc', 'n_events_t1', 'max_rise_rate_t1', 'drawdown_index_t1', 'flood_days_t1', 'n_events_t2', 'max_rise_rate_t2', 'drawdown_index_t2', 'flood_days_t2', 'bend_exposure_n5', 'bend_exposure_n8', 'split', 'cluster']

Null counts (region_features):
vegetation_class    2024
land_use            4573
soil_group           805
Saved → /Users/admin/Documents/work/Moraine/oevererosie/wocu-

## Step 4 — Train models

Trains Naive / v_train / OLS / Ridge (numeric) / Ridge (+cat) / LightGBM
and saves a bundle to `04_model_outputs/{EXPERIMENT}/`.

In [7]:
print('Running Step 4: training models ...')
results = train_and_save_models(region_features, MODEL_OUT_DIR, seed=SEED)

print('\n── Model comparison (test MAE) ──')
comparison = pd.DataFrame(results).T[['test_mae', 'test_rmse', 'test_r2', 'test_tail_mae']]
comparison.sort_values('test_mae')

Running Step 4: training models ...

── 0 – Naive mean
          metric    train       test
      MAE (prim)    0.7976    0.7843
            RMSE    2.1832    2.1993
      MAE tail>2    5.4531    5.5200  (n=325/78)
              R²    0.0000    -0.0000

── 1 – v_train baseline
          metric    train       test
      MAE (prim)    1.1562    1.2240
            RMSE    3.0229    3.2509
      MAE tail>2    6.5363    6.9339  (n=325/78)
              R²    -0.9172    -1.1850

── 2 – OLS v_train
          metric    train       test
      MAE (prim)    0.8144    0.8447
            RMSE    2.0239    2.0888
      MAE tail>2    4.9052    4.7493  (n=325/78)
              R²    0.1406    0.0979

── 3 – Ridge numeric
          metric    train       test
      MAE (prim)    0.8440    0.8718
            RMSE    2.0089    2.0733
      MAE tail>2    4.8135    4.6583  (n=325/78)
              R²    0.1532    0.1113

── 4 – Ridge + cat
          metric    train       test
      MAE (prim)    0.8455    

,test_mae,test_rmse,test_r2,test_tail_mae
0 – Naive mean,0.784335,2.199337,-0.000046,5.519952
5 – LightGBM,0.801435,2.050304,0.130893,4.803066
2 – OLS v_train,0.844653,2.088848,0.097909,4.749319
4 – Ridge + cat,0.871693,2.061210,0.121623,4.624609
3 – Ridge numeric,0.871771,2.073259,0.111324,4.658301
1 – v_train baseline,1.224000,3.250923,-1.184991,6.933882


## Step 5 — Iterative prediction & export

Loads the trained LGB bundle, runs rolling year-by-year predictions for
all 8k+ locations (2026–2050), computes VVR crossing years,
and exports a full GeoPackage.

In [8]:
# ── Load inputs ───────────────────────────────────────────────────────────────
bundle           = load_model_bundle(MODEL_OUT_DIR)
inference_df     = pd.read_parquet(FEATURES_DIR / 'region_inference_features.parquet')
region_split_df  = pd.read_parquet(FEATURES_DIR / 'region_split.parquet')
inf_meta         = pd.read_parquet(FEATURES_DIR / 'region_inference_only.parquet')

print(f'Bundle loaded from : {MODEL_OUT_DIR}')
print(f'LGB features       : {bundle["config"]["FEATS_LGB"]}')

Bundle loaded from : /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/data/04_model_outputs/20260617a
LGB features       : ['v_train', 'dist_t2', 'train_span_yr', 'test_span_yr', 'erosion_vol_rate_t1', 'n_events_t1', 'max_rise_rate_t1', 'drawdown_index_t1', 'flood_days_t1', 'n_events_t2', 'max_rise_rate_t2', 'drawdown_index_t2', 'flood_days_t2', 'bend_exposure_n5', 'bend_exposure_n8', 'is_nvo', 'river_enc', 'vegetation_class_enc', 'soil_group_enc', 'land_use_enc']


In [9]:
# ── Build start points ────────────────────────────────────────────────────────
start_points = {}

for loc_id, row in region_features.iterrows():
    t3_year = int(region_split_df.loc[loc_id, 't3']) if loc_id in region_split_df.index else 2025
    start_points[loc_id] = {
        'last_dist': row['dist_t3'],
        'last_year': t3_year,
        'v_hist':    row['v_train'],
    }

for loc_id, row in inference_df.iterrows():
    if loc_id not in start_points:
        t2_year = int(inf_meta.loc[loc_id, 't2']) if loc_id in inf_meta.index else 2022
        start_points[loc_id] = {
            'last_dist': row['dist_t2'],
            'last_year': t2_year,
            'v_hist':    row['v_train'],
        }

print(f'Total start_points: {len(start_points):,}')

Total start_points: 8,094


In [10]:
# ── Combined feature table ────────────────────────────────────────────────────
inference_df_filled = inference_df.copy()
if 'test_span_yr' not in inference_df_filled.columns:
    inference_df_filled['test_span_yr'] = inference_df_filled['train_span_yr']

shared_cols = [c for c in region_features.columns if c in inference_df_filled.columns]
all_features = pd.concat([region_features[shared_cols], inference_df_filled[shared_cols]])

print(f'Combined feature table: {all_features.shape}')

# ── Run rolling LGB prediction ────────────────────────────────────────────────
print(f'Running LGB rolling prediction ({START_YEAR}–{END_YEAR}) ...')
df_lgb = predict_iterative_ml(
    bundle=bundle,
    features_df=all_features,
    start_points=start_points,
    model_name='lgb',
    start_year=START_YEAR,
    end_year=END_YEAR,
    step=1,
    rolling=True,
)
print(f'Predictions: {len(df_lgb):,} rows  ({df_lgb["location_id"].nunique():,} locations)')

Combined feature table: (8094, 26)
Running LGB rolling prediction (2026–2050) ...
Predictions: 202,350 rows  (8,094 locations)


In [11]:
# ── Load geometric inputs ─────────────────────────────────────────────────────
print('Loading geometric inputs ...')
scope_raw   = ensure_location_id_column(gpd.read_file(RAW_GPKG,      layer='vlakken_scope'))
centerlines = ensure_location_id_column(gpd.read_file(RAW_GPKG,      layer='centrelines'))
bank_points = ensure_location_id_column(gpd.read_file(RAW_GPKG,      layer='punten_oever'))
vvr         = gpd.read_file(PROC_GPKG, layer='vvr_rates_of_change')
scope       = ensure_location_id_column(gpd.read_file(PROC_GPKG,     layer='summary_scope'))
signalering = gpd.read_file(SIGNALERING_GPKG, layer=SIGNALERING_LAYER).to_crs(28992)

cl_lookup       = centerlines.set_index('location_id')['geometry'].to_dict()
ref_geom_lookup = build_ref_geom_lookup(bank_points, n_points=N_POINTS)
nvo_ids         = get_nvo_location_ids(vvr, scope)

print(f'Centerlines: {len(cl_lookup):,}  |  NVO locations: {len(nvo_ids):,}')

Loading geometric inputs ...
Centerlines: 12,130  |  NVO locations: 2,925


In [ ]:
# ── Build predicted bank positions ────────────────────────────────────────────
records = []
for row in df_lgb.itertuples(index=False):
    cline = cl_lookup.get(row.location_id)
    ref   = ref_geom_lookup.get(row.location_id)
    point = (
        point_from_offset(cline, row.predicted_dist_m, ref)
        if cline is not None and ref is not None and len(ref) > 0
        else None
    )
    records.append({
        'location_id':       row.location_id,
        'year':              row.year,
        'predicted_dist_m':  row.predicted_dist_m,
        'velocity_m_per_yr': row.velocity_m_per_yr,
        'is_nvo':            int(row.location_id in nvo_ids),
        'geometry':          point,
    })

predicted_bank_positions = gpd.GeoDataFrame(records, geometry='geometry', crs=centerlines.crs)
valid = predicted_bank_positions.geometry.notna().sum()
print(f'predicted_bank_positions: {len(predicted_bank_positions):,} rows  ({valid:,} with geometry)')

predicted_bank_positions: 202,350 rows  (202,350 with geometry)


In [13]:
# ── VVR crossing years ────────────────────────────────────────────────────────
vvr_crossing = compute_vvr_crossing_year(
    df_lgb, nvo_ids, centerlines, scope_raw, signalering,
    reference_year=START_YEAR - 1,
)
crossed = vvr_crossing[vvr_crossing['crossing_year'].notna()]
print(f'VVR crossing: {len(crossed):,} crossings within {START_YEAR}–{END_YEAR}')

VVR crossing: 281 crossings within 2026–2050


In [14]:
# ── Export GeoPackage ─────────────────────────────────────────────────────────
OUTPUT_GPKG = MODEL_OUT_DIR / f'wocu_lgb_predictions_{EXPERIMENT}.gpkg'

output_path = export_predictions(
    base_gpkg=PROC_GPKG,
    output_gpkg=OUTPUT_GPKG,
    predicted_bank_positions=predicted_bank_positions,
    vvr_crossing=vvr_crossing,
    scope_raw=scope_raw,
    signaleringslijn=signalering,
)
print(f'Exported → {output_path}  ({output_path.stat().st_size / 1e6:.1f} MB)')

[1/5] Copying base GPKG → wocu_lgb_predictions_20260617a.gpkg ...
      done  (166.6 MB)  0.1s
[2/5] Writing predicted_bank_positions (202,350 rows) ...
      to_file done  1.0s  (26.1 MB)
      sqlite ATTACH copy done  0.1s
[3/5] Computing crossing year per VVR polygon ...
      done  0.0s  → 278 crossing  879 safe (=9999)  253 no-prediction (NULL)
[4/5] Patching vvr_rates_of_change.predicted_vvr_crossing_year ...
      done  0.0s
[5/5] Writing signaleringslijn (529 features) ...
      done  0.0s
Exported → /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/data/04_model_outputs/20260617a/wocu_lgb_predictions_20260617a.gpkg  (184.7 MB)


## Acceptance criteria

In [15]:
print('=== Acceptance criteria ===')
passed = []

# 1. Location counts
n_pred = predicted_bank_positions['location_id'].nunique()
n_expected = len(region_features) + len(inference_df)
c1 = n_pred >= len(region_features)
passed.append(c1)
print(f'[{"OK" if c1 else "FAIL"}] 1. Unique prediction locations: {n_pred:,}  (expected ~{n_expected:,})')

# 2. Required layers
layers_in_gpkg = {name for name, _ in list_layers(output_path)}
required = {'predicted_bank_positions', 'vvr_rates_of_change', 'summary_scope', 'signaleringslijn'}
missing = required - layers_in_gpkg
c2 = len(missing) == 0
passed.append(c2)
print(f'[{"OK" if c2 else "FAIL"}] 2. Required layers present{(": MISSING " + str(missing)) if missing else ""}')

# 3. is_nvo is integer
c3 = predicted_bank_positions['is_nvo'].dtype in [int, 'int64', 'int32']
passed.append(c3)
print(f'[{"OK" if c3 else "FAIL"}] 3. is_nvo dtype: {predicted_bank_positions["is_nvo"].dtype}')

# 4. VVR crossing column present
vvr_out = gpd.read_file(output_path, layer='vvr_rates_of_change')
c4 = 'predicted_vvr_crossing_year' in vvr_out.columns
passed.append(c4)
n_filled = vvr_out['predicted_vvr_crossing_year'].notna().sum() if c4 else 0
print(f'[{"OK" if c4 else "FAIL"}] 4. predicted_vvr_crossing_year: {n_filled:,}/{len(vvr_out):,} filled')

print(f'\n{sum(passed)}/{len(passed)} checks passed.')
if all(passed):
    print('✓ ALL ACCEPTANCE CRITERIA MET')
else:
    print('✗ Some checks failed — review above')

=== Acceptance criteria ===
[OK] 1. Unique prediction locations: 8,094  (expected ~8,094)
[OK] 2. Required layers present
[OK] 3. is_nvo dtype: int64
[OK] 4. predicted_vvr_crossing_year: 1,157/1,410 filled

4/4 checks passed.
✓ ALL ACCEPTANCE CRITERIA MET
